##Setup and Data Download
This initial cell remains the same. It sets up the environment, downloads the dataset, and sets a random seed for reproducibility.

In [1]:
# Install libraries
!pip install -q albumentations

# Imports
import torch
import os
import random
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# === DIRECT KAGGLE CREDENTIALS ===
os.environ['KAGGLE_USERNAME'] = 'asd147'
os.environ['KAGGLE_KEY']     = '6a95e405001115800e2e18044513a965'
# ===================================

# Download and unzip data
!kaggle datasets download -d andrewmvd/car-plate-detection
!unzip -q -o car-plate-detection.zip -d ./data
!rm -f car-plate-detection.zip

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Dataset and Transform Functions (from previous step) ---
IMG_WIDTH, IMG_HEIGHT = 512, 512

def get_transform(train):
    # Same optimized transforms
    # ... [Copy-paste the get_transform function from the previous step]
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.75),
            A.GaussianBlur(blur_limit=(3, 5), p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))
    else: # For validation, we ONLY normalize and convert to tensor
        return A.Compose([
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

class PlateAlbumentationsDataset(Dataset):
    # ... [Copy-paste the PlateAlbumentationsDataset class from the previous step]
    def __init__(self, data_dir, image_files, width, height, transforms=None):
        self.transforms = transforms
        self.data_dir = data_dir
        self.height = height
        self.width = width
        self.image_files = image_files

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.data_dir, 'images', img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_resized = cv2.resize(image, (self.width, self.height), interpolation=cv2.INTER_LANCZOS4)

        ann_name = img_name.replace('.png', '.xml')
        ann_path = os.path.join(self.data_dir, 'annotations', ann_name)
        tree = ET.parse(ann_path)
        root = tree.getroot()

        orig_w = int(root.find('size').find('width').text)
        orig_h = int(root.find('size').find('height').text)
        bndbox = root.find('object').find('bndbox')

        xmin = (float(bndbox.find('xmin').text) / orig_w) * self.width
        ymin = (float(bndbox.find('ymin').text) / orig_h) * self.height
        xmax = (float(bndbox.find('xmax').text) / orig_w) * self.width
        ymax = (float(bndbox.find('ymax').text) / orig_h) * self.height

        boxes = [[xmin, ymin, xmax, ymax]]
        labels = [1]
        target = {'boxes': boxes, 'labels': labels}

        if self.transforms:
            augmented = self.transforms(image=image_resized, bboxes=target['boxes'], labels=target['labels'])
            image_tensor = augmented['image']
            target['boxes'] = torch.as_tensor(augmented['bboxes'], dtype=torch.float32) if augmented['bboxes'] else torch.zeros((0, 4), dtype=torch.float32)
            target['labels'] = torch.as_tensor(augmented['labels'], dtype=torch.int64)

        # Return original resized image for visualization and tensor for training
        return image_resized, image_tensor, target

    def __len__(self):
        return len(self.image_files)

def collate_fn_ensemble(batch):
    # Modified collate to handle the extra raw image
    raw_images, images, targets = zip(*batch)
    return list(raw_images), list(images), list(targets)

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/car-plate-detection
License(s): CC0-1.0
100% 203M/203M [00:15<00:00, 13.9MB/s]

Using device: cuda


##Training the Ensemble of Models
We will now loop to train three separate models, each with a different random seed.

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def create_model():
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
    return model

num_models = 3
ensemble_models = []
seeds = [42, 84, 126] # Different seeds for diversity
data_dir = './data'
all_image_files = sorted([f for f in os.listdir(os.path.join(data_dir, 'images')) if f.endswith('.png')])

for i in range(num_models):
    print(f"--- Training Model {i+1}/{num_models} with Seed {seeds[i]} ---")
    set_seed(seeds[i])

    # Create new data split for each model
    train_files, val_files = train_test_split(all_image_files, test_size=0.2, random_state=seeds[i])
    train_dataset = PlateAlbumentationsDataset(data_dir, train_files, IMG_WIDTH, IMG_HEIGHT, get_transform(train=True))
    train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True, collate_fn=lambda b: tuple(zip(*b))[1:]) # Only need tensors for training

    model = create_model().to(device)

    # Optimizer and scheduler
    num_epochs = 20 # Can reduce epochs slightly as ensembling is powerful
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.SGD(params, lr=0.003, momentum=0.9, weight_decay=0.0005)
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0.00001)

    for epoch in range(num_epochs):
        model.train()
        for images, targets in tqdm(train_loader, desc=f"Model {i+1} Epoch {epoch+1}"):
            valid_batch = [(img, tgt) for img, tgt in zip(images, targets) if len(tgt['boxes']) > 0]
            if not valid_batch: continue
            images_b, targets_b = zip(*valid_batch)
            images_b = list(img.to(device) for img in images_b)
            targets_b = [{k: v.to(device) for k, v in t.items()} for t in targets_b]
            loss_dict = model(images_b, targets_b)
            losses = sum(loss for loss in loss_dict.values())
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
        lr_scheduler.step()

    ensemble_models.append(model)
    print(f"--- Finished Training Model {i+1} ---")

--- Training Model 1/3 with Seed 42 ---
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:02<00:00, 82.5MB/s]
Model 1 Epoch 20: 100%|██████████| 58/58 [01:32<00:00,  1.60s/it]


--- Finished Training Model 1 ---
--- Training Model 2/3 with Seed 84 ---


Model 2 Epoch 20: 100%|██████████| 58/58 [01:32<00:00,  1.60s/it]


--- Finished Training Model 2 ---
--- Training Model 3/3 with Seed 126 ---


Model 3 Epoch 20: 100%|██████████| 58/58 [01:32<00:00,  1.59s/it]

--- Finished Training Model 3 ---


##Evaluation with Ensemble and TTA
This is the most critical part. We create a dedicated validation loader and then loop through it, applying our advanced prediction logic.

In [3]:
def calculate_iou_detection(pred_boxes, true_boxes):
    # ... [Copy-paste the calculate_iou_detection function from the previous step]
    if pred_boxes.shape[0] == 0 or true_boxes.shape[0] == 0: return torch.tensor(0.0)
    pred_boxes, true_boxes = pred_boxes.view(-1, 4), true_boxes.view(-1, 4)
    inter_xmin = torch.max(pred_boxes[:, 0].unsqueeze(1), true_boxes[:, 0])
    inter_ymin = torch.max(pred_boxes[:, 1].unsqueeze(1), true_boxes[:, 1])
    inter_xmax = torch.min(pred_boxes[:, 2].unsqueeze(1), true_boxes[:, 2])
    inter_ymax = torch.min(pred_boxes[:, 3].unsqueeze(1), true_boxes[:, 3])
    inter_area = torch.clamp(inter_xmax - inter_xmin, min=0) * torch.clamp(inter_ymax - inter_ymin, min=0)
    pred_area = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (pred_boxes[:, 3] - pred_boxes[:, 1])
    true_area = (true_boxes[:, 2] - true_boxes[:, 0]) * (true_boxes[:, 3] - true_boxes[:, 1])
    union_area = pred_area.unsqueeze(1) + true_area - inter_area
    iou = inter_area / (union_area + 1e-6)
    return iou.max(dim=1)[0]

# Use the initial seed for a consistent validation set for final scoring
set_seed(42)
_, val_files = train_test_split(all_image_files, test_size=0.2, random_state=42)
val_dataset = PlateAlbumentationsDataset(data_dir, val_files, IMG_WIDTH, IMG_HEIGHT, get_transform(train=False))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn_ensemble)

# Set all models to evaluation mode
for model in ensemble_models:
    model.eval()

total_iou = 0.0
total_images = 0

with torch.no_grad():
    for raw_images, images, targets in tqdm(val_loader, desc="Evaluating with Ensemble + TTA"):
        images = list(img.to(device) for img in images)

        for i in range(len(images)):
            ensemble_boxes = []

            # Get predictions from all models
            for model in ensemble_models:
                # --- TTA Part 1: Original Image ---
                output_orig = model([images[i]])[0]
                if len(output_orig['boxes']) > 0:
                    ensemble_boxes.append(output_orig['boxes'][0].cpu())

                # --- TTA Part 2: Flipped Image ---
                img_flipped = torch.flip(images[i], dims=[2])
                output_flipped = model([img_flipped])[0]
                if len(output_flipped['boxes']) > 0:
                    box_flipped = output_flipped['boxes'][0].cpu()
                    # Un-flip the box coordinates
                    unflipped_box = torch.tensor([
                        IMG_WIDTH - box_flipped[2], # new x_min
                        box_flipped[1],             # y_min
                        IMG_WIDTH - box_flipped[0], # new x_max
                        box_flipped[3]              # y_max
                    ])
                    ensemble_boxes.append(unflipped_box)

            if not ensemble_boxes: continue

            # --- Ensemble Part: Average the collected boxes ---
            final_box = torch.stack(ensemble_boxes).mean(dim=0).unsqueeze(0)

            true_boxes = targets[i]['boxes'].to('cpu')
            iou_score = calculate_iou_detection(final_box, true_boxes)
            total_iou += iou_score.sum().item()

        total_images += len(images)

avg_iou = total_iou / total_images
print(f"\nFINAL IoU with Ensemble + TTA: {avg_iou:.4f}")

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
Evaluating with Ensemble + TTA: 100%|██████████| 22/22 [00:50<00:00,  2.29s/it]


FINAL IoU with Ensemble + TTA: 0.7943
